# Report Q4 2025's Meta ROAS

*Data Analysis in Python · Topic 5.4*

The social team's draft says Meta returned almost $5 for every dollar in the last quarter of 2025. Check the join behind that number, and report Meta's ROAS for October to December 2025, overall and by campaign.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `sessions` table (lesson 3.'s steps), the cell that builds the `revenue` table (topic 5.2.'s steps) and the cell that builds the `spend` table: Meta's spend for October to December 2025, in topic 5.1.'s standard columns, with topic 5.3.'s `campaign_key()` applied to both tables.
2. In the first cell marked ✏️, create:
   - the `naive` table: the `spend` table joined to the `revenue` table on the date and the campaign, keeping every row of the `spend` table
   - `naive_roas`: the `naive` table's revenue over its spend, the social team's number
   - `repeated_keys`: the number of rows of the `spend` table that repeat an earlier row's date and campaign
3. In the second cell marked ✏️, create:
   - the `meta_daily` table: one row per date and campaign of the `spend` table, with the columns `date`, `campaign`, `impressions`, `clicks` and `spend`
   - the `joined` table: the `meta_daily` table joined to the `revenue` table like the `naive` one, with `validate=` set so that `merge()` stops if a date and campaign appear more than once on either side
   - `meta_roas`: the `joined` table's revenue over its spend
4. In the last cell marked ✏️, create:
   - the `by_campaign` table: one row per campaign, with the campaigns as the index, and the columns `spend`, `revenue` and `roas`, sorted by `roas`, highest first
   - `best_campaign`: the name of the campaign with the highest ROAS
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `naive_roas` is about `4.8050`, and `repeated_keys` is `1008`. The `meta_daily` table has 640 rows and the same $26,410.31 of spend. The `joined` table has 640 rows, and `meta_roas` is about `1.8979`. The `by_campaign` table has 12 campaigns, sorted by ROAS, and `best_campaign` is `"2025_ca_instagram_blackfriday"`.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-10", "2025-11", "2025-12"]
START, END = "2025-10-01", "2025-12-31"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# Revenue per date and campaign (topic 5.2) and Meta's spend (topic 5.1), with the keys fixed (topic 5.3).
# Run it, and leave it as it is.
def campaign_key(names):
    """Return campaign names stripped of spaces and lowercased, so they match across systems."""
    return names.str.strip().str.lower()


revenue = sessions.groupby(["date", "session_campaign"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = campaign_key(revenue["campaign"])

meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts"])
spend = meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Ad set name": "ad_set",
                             "Impressions": "impressions", "Link clicks": "clicks", "Amount spent (USD)": "spend"})
spend["platform"] = "Meta"
spend["campaign"] = campaign_key(spend["campaign"])
spend = spend[spend["date"].between(START, END)].reset_index(drop=True)
spend = spend[["date", "platform", "campaign", "ad_set", "impressions", "clicks", "spend"]]
spend.shape, revenue.shape

## The social team's number

In [ ]:
# ✏️ Create naive, naive_roas and repeated_keys


## One row per campaign and day

In [ ]:
# ✏️ Create meta_daily, joined and meta_roas


## Campaigns

In [ ]:
# ✏️ Create by_campaign and best_campaign


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    naive = found.get("naive")
    naive_roas = found.get("naive_roas")
    repeated_keys = found.get("repeated_keys")
    if not isinstance(naive, pd.DataFrame) or len(naive) != 1648:
        results.append((False, "naive should be the spend table joined to the revenue table, with the spend table's 1,648 rows: spend.merge(revenue, on=[\"date\", \"campaign\"], how=\"left\")."))
    elif not pd.api.types.is_number(naive_roas) or isinstance(naive_roas, (pd.DataFrame, pd.Series)) or abs(naive_roas - 4.8050) > 0.0001:
        results.append((False, f"naive_roas should be the naive table's revenue total over its spend total, about 4.8050: the social team's number. It is {naive_roas!r}."))
    elif isinstance(repeated_keys, (pd.DataFrame, pd.Series)):
        results.append((False, f"repeated_keys is a {type(repeated_keys).__name__}, not a number. Count the repeats: spend.duplicated([\"date\", \"campaign\"]).sum()."))
    elif repeated_keys == 0:
        results.append((False, "repeated_keys is 0: that's every column of the row compared. Compare the date and the campaign only: spend.duplicated([\"date\", \"campaign\"]).sum()."))
    elif repeated_keys == 1648:
        results.append((False, "repeated_keys is 1,648: every row with a repeated key, the first one included. duplicated() counts the repeats only: 1,008."))
    elif repeated_keys != 1008:
        results.append((False, f"repeated_keys should be spend.duplicated([\"date\", \"campaign\"]).sum(), 1,008. It is {repeated_keys!r}."))
    else:
        results.append((True, "The social team's 4.81 comes from 1,648 ad-set rows joined to 640 campaign-days: 1,008 rows repeat a key."))

    meta_daily = found.get("meta_daily")
    joined = found.get("joined")
    meta_roas = found.get("meta_roas")
    if not isinstance(meta_daily, pd.DataFrame):
        results.append((False, "meta_daily isn't a DataFrame yet. Group the spend table by date and campaign with as_index=False, and total impressions, clicks and spend."))
    elif "ad_set" in meta_daily.columns or len(meta_daily) == 1648:
        results.append((False, f"The meta_daily table has {len(meta_daily):,} rows: it's still one row per ad set. Group by date and campaign only."))
    elif len(meta_daily) != 640 or not {"date", "campaign", "spend"} <= set(meta_daily.columns):
        results.append((False, f"The meta_daily table should have 640 rows, one per date and campaign, with date and campaign as columns. It has {len(meta_daily):,} rows and {list(meta_daily.columns)}."))
    elif abs(meta_daily["spend"].sum() - 26410.31) > 0.01:
        results.append((False, f"The meta_daily table's spend adds up to {meta_daily['spend'].sum():,.2f}, but the spend table's is 26,410.31. Total spend with sum()."))
    elif not isinstance(joined, pd.DataFrame) or len(joined) != 640:
        results.append((False, f"The joined table should be the meta_daily table joined to the revenue table, 640 rows. It has {len(joined) if isinstance(joined, pd.DataFrame) else 'no'} rows."))
    elif history is not None and not any("validate=\"one_to_one\"" in source.replace("'", "\"").replace(" ", "") or "validate=\"1:1\"" in source.replace("'", "\"").replace(" ", "") for source in history):
        results.append((False, "The joined table is right, but no merge in your notebook says validate=\"one_to_one\". Add it, so the merge stops if a key repeats on either side."))
    elif not pd.api.types.is_number(meta_roas) or isinstance(meta_roas, (pd.DataFrame, pd.Series)):
        results.append((False, f"meta_roas should be the joined table's revenue total over its spend total. It is {meta_roas!r}."))
    elif abs(meta_roas - 4.8050) < 0.0001:
        results.append((False, "meta_roas is 4.81, the naive number: it's from the ad-set rows. Use the joined table, made from meta_daily."))
    elif abs(meta_roas - 1.8979) > 0.0001:
        results.append((False, f"meta_roas should be the joined table's revenue over its spend, about 1.8979. It is {meta_roas!r}."))
    else:
        results.append((True, f"Meta's ROAS for Q4 2025 is {meta_roas:.2f}, not 4.81: each campaign-day's revenue counted once."))

    by_campaign = found.get("by_campaign")
    best = found.get("best_campaign")
    if not isinstance(by_campaign, pd.DataFrame):
        results.append((False, "by_campaign isn't a DataFrame yet. Group the joined table by campaign, total spend and revenue, and add roas after totaling."))
    elif len(by_campaign) != 12 or "2025_us_facebook_holiday" not in by_campaign.index:
        results.append((False, f"The by_campaign table should have the 12 campaigns as its index. It has {len(by_campaign):,} rows."))
    elif not {"spend", "revenue", "roas"} <= set(by_campaign.columns):
        results.append((False, f"The by_campaign table should have the columns spend, revenue and roas. It has {list(by_campaign.columns)}."))
    elif abs(by_campaign["revenue"].sum() - 126901.38) < 0.01:
        results.append((False, "The by_campaign table's revenue adds up to $126,901.38: it's grouped from the naive table. Group the joined table."))
    elif abs(by_campaign["revenue"].sum() - 50124.71) > 0.01 or not ((by_campaign["roas"] - by_campaign["revenue"] / by_campaign["spend"]).abs() < 1e-9).all():
        results.append((False, "The by_campaign table's roas should be each campaign's revenue over its spend, totaled from the joined table."))
    elif by_campaign.index[0] != "2025_ca_instagram_blackfriday" or not by_campaign["roas"].is_monotonic_decreasing:
        results.append((False, f"The by_campaign table should be sorted by roas, highest first. Its first row is {by_campaign.index[0]}."))
    elif best != "2025_ca_instagram_blackfriday":
        results.append((False, f"best_campaign should be the name of the first campaign in the sorted table: by_campaign.index[0]. It is {best!r}."))
    else:
        results.append((True, "2025_ca_instagram_blackfriday leads with 4.42, and 2025_us_facebook_blackfriday makes $1.00 for each dollar."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
naive = spend.merge(revenue, on=["date", "campaign"], how="left")
naive_roas = naive["revenue"].sum() / naive["spend"].sum()
repeated_keys = spend.duplicated(["date", "campaign"]).sum()
naive_roas, repeated_keys
```

```python
meta_daily = spend.groupby(["date", "campaign"], as_index=False)[["impressions", "clicks", "spend"]].sum()
joined = meta_daily.merge(revenue, on=["date", "campaign"], how="left", validate="one_to_one")
meta_roas = joined["revenue"].sum() / joined["spend"].sum()
len(meta_daily), meta_roas
```

```python
by_campaign = joined.groupby("campaign")[["spend", "revenue"]].sum()
by_campaign["roas"] = by_campaign["revenue"] / by_campaign["spend"]
by_campaign = by_campaign.sort_values("roas", ascending=False)
best_campaign = by_campaign.index[0]
by_campaign
```

</details>